# NB03: Data Analysis

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214

## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
!pip install python-dotenv
!pip install pyperclip
!pip install tabulate

import os
import json
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import pyperclip
import tabulate
import plotly.express as px

## The desired graphs

Now that our dataset has been transformed, it is time to transform the raw data into graphs to answer the question of whether Pokemon has experienced power creep over the years, and if so, to what extent it has been experienced. I am going to build multiple different graphs: a line graph with the overall average BST over each of the generations; a line graph with the average BST by my three chosen categories of Pokemon (Standard, Legendary, Mythical); a histogram showing the overall distribution of BSTs, with each generation having its own bar marked by color; histograms showing the overall distribution of BSTs for my three chosen categories of Pokemon; a line graph showing the average individual base stats (hp, attack, defense, special attack, special defense, speed) by generation, each like separated by color; a segregated line graph showing the average individual base stats for my three chosen categories of Pokemon; histograms showing the overall distribution of each of the individual base stats, with each generation having its own color of bar; and histograms showing the overall distribution for each of the base stats for my three chosen categories of Pokemon, which each generation being marked by its own color of bar. 

I have chosen to use line graphs for some of my charts because I am comparing averages over a period of time (different Generations), and since line graphs are best for displaying data over time, they are my best choice here. As for why I am choosing histograms, I am using them because they display a count of all of the objects that fall into certain bins, which makes them better than bar charts for me because Pokemon BSTs and individual base stats are not always nice round numbers, so being able to group them into bins made up of round numbers will make the data cleaner.

## Creating the graphs

I am going to start by creating all of the graphs pertaining to average BSTs. Though the data has been properly transformed, I still need to create new data frames that have my values aggregated in the way that I want them to be.

### Line graph of average BST over generations

In [2]:
#First, loading the csv

df_comp = pd.read_csv('../data/processed/complete_table.csv')

df_comp

,is_default,pokemon_name,pokemon_url,id_x,name,is_baby,is_legendary,is_mythical,forms_switchable,has_gender_differences,gen_x,gen_url,BST,base_stat,stat_name,id_y,gen_y,gen_num
0,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,45.0,hp,1.0,generation-i,Gen 1
1,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,49.0,attack,1.0,generation-i,Gen 1
2,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,49.0,defense,1.0,generation-i,Gen 1
3,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,65.0,special-attack,1.0,generation-i,Gen 1
4,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,65.0,special-defense,1.0,generation-i,Gen 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5960,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,80.0,attack,1019.0,generation-ix,Gen 9
5961,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,110.0,defense,1019.0,generation-ix,Gen 9
5962,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,120.0,special-attack,1019.0,generation-ix,Gen 9
5963,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,80.0,special-defense,1019.0,generation-ix,Gen 9


In [3]:
dfg = (
    df_comp
    .groupby(['gen_num'], as_index = False)
    ['BST'].mean()
)

dfg

,gen_num,BST
0,Gen 1,407.642384
1,Gen 2,407.180000
2,Gen 3,402.506832
3,Gen 4,442.641148
4,Gen 5,420.912458
5,Gen 6,425.248756
6,Gen 7,451.328032
7,Gen 8,434.889094
8,Gen 9,456.785507


In [4]:
avg_bst_lg = px.line(
    dfg,
    x = 'gen_num',
    y = 'BST',
    title = 'Over generations, average BSTs of all Pokemon have increased since Generation 4, though not at a consistent rate',
    subtitle = 'The BSTs were calculated by adding up all of a Pokemon\'s base stats.',
    labels = {
        'gen_num':'Generation',
        'BST':'Average Base Stat Totals'
    },
    markers = True
)

avg_bst_lg.update_yaxes(rangemode='tozero')

avg_bst_lg.show()

In [5]:
# Needs to be moved to NB02

def mon_class(row):
    if row['is_legendary'] == False and row['is_mythical'] == False:
        return 'Standard'
    elif row['is_legendary'] == True and row['is_mythical'] == False:
        return 'Legendary'
    elif row['is_legendary']== False and row['is_mythical'] == True:
        return 'Mythical'

In [6]:
df_comp['Classification'] = df_comp.apply(lambda row: mon_class(row), axis = 1)

df_comp

,is_default,pokemon_name,pokemon_url,id_x,name,is_baby,is_legendary,is_mythical,forms_switchable,has_gender_differences,gen_x,gen_url,BST,base_stat,stat_name,id_y,gen_y,gen_num,Classification
0,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,45.0,hp,1.0,generation-i,Gen 1,Standard
1,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,49.0,attack,1.0,generation-i,Gen 1,Standard
2,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,49.0,defense,1.0,generation-i,Gen 1,Standard
3,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,65.0,special-attack,1.0,generation-i,Gen 1,Standard
4,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,65.0,special-defense,1.0,generation-i,Gen 1,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5960,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,80.0,attack,1019.0,generation-ix,Gen 9,Standard
5961,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,110.0,defense,1019.0,generation-ix,Gen 9,Standard
5962,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,120.0,special-attack,1019.0,generation-ix,Gen 9,Standard
5963,True,hydrapple,https://pokeapi.co/api/v2/pokemon/1019/,1019,hydrapple,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,540,80.0,special-defense,1019.0,generation-ix,Gen 9,Standard


In [7]:
dfg = (
    df_comp
    .groupby(['Classification','gen_num'], as_index = False)
    ['BST'].mean()
)

dfg

,Classification,gen_num,BST
0,Legendary,Gen 1,605.000000
1,Legendary,Gen 2,620.000000
2,Legendary,Gen 3,620.000000
3,Legendary,Gen 4,620.000000
4,Legendary,Gen 5,623.589744
5,Legendary,Gen 6,673.846154
6,Legendary,Gen 7,540.363636
7,Legendary,Gen 8,578.928571
8,Legendary,Gen 9,571.363636
9,Mythical,Gen 1,600.000000


In [12]:
class_bst_lg = px.line(
    dfg,
    x = 'gen_num',
    y = 'BST',
    color = 'Classification',
    title = 'Over generations, average BSTs of Standard Pokemon have tended to increase, though not at a consistent rate',
    subtitle = 'Legendaries\' BSTs have stayed consistent until Generation 7, when they decreased. Mythicals\' BSTs have been very consistent outside of a Generation 7 dip.',
    labels = {
        'gen_num':'Generation',
        'BST':'Average Base Stat Totals'
    },
    markers = True
)

class_bst_lg.update_yaxes(rangemode='tozero')

class_bst_lg.show()